# Projekt 03 (final): Spamfilter mit Naive Bayes — auf echten Daten

**Ziel:** Du baust einen kompletten Spam-Klassifikator **von Grund auf selbst** —
mit nichts als der Bayes-Regel, Zaehlen und Logarithmen — und misst ihn an echten
Daten: 5.574 echten SMS (davon ~13 % Spam) aus der *SMS Spam Collection* (UCI).
Am Ende vergleichst du dein Modell mit der scikit-learn-Implementierung.

**Vorbereitung:** Einmalig im Terminal (aus dem Ordner `03-final`, venv aktiv):

```
python datasets/download_data.py
```

**Bezug zum Skript:** Abschnitt 2.4 (Bayes-Regel, Naive Bayes) und 2.5 (Supervised Learning).

## 1. Daten laden und anschauen

Regel Nummer eins bei echten Daten: **erst anschauen, dann modellieren**.

**Aufgabe:** Lade `datasets/SMSSpamCollection` (Tab-getrennt, keine Kopfzeile, Spalten `label` und `text`; `pd.read_csv(..., sep="\t", header=None, names=["label","text"], quoting=3)` — `quoting=3` verhindert, dass " als Anfuehrungszeichen gedeutet wird). Verschaff dir dann einen Ueberblick: Klassenverteilung (`value_counts`), Spam-Anteil (~13 %) und vergleiche die Textlaenge von ham vs. spam (`str.len`, `groupby`).


In [ ]:
# Dein Code hier. (Musterloesung: solution/solution.ipynb)


In [ ]:
# Dein Code hier. (Musterloesung: solution/solution.ipynb)


**Wichtig — unausgewogene Klassen:** Nur ~13 % Spam. Ein "Klassifikator", der stur
*ham* sagt, haette schon ~87 % Accuracy! Accuracy allein reicht hier also nicht als
Erfolgsmass — das pruefen wir spaeter mit Precision und Recall nach.

## 2. Trainings- und Testmenge

Wir bewerten das Modell nur auf SMS, die es beim Training **nie gesehen** hat
(Skript 2.5: Generalisierung!). `stratify` sorgt dafuer, dass der Spam-Anteil in
beiden Teilmengen gleich ist; der feste `random_state` macht alles reproduzierbar.

**Aufgabe:** Teile mit `train_test_split` in 80 % Training / 20 % Test (`test_size=0.2`, `random_state=42`, `stratify=labels`) und pruefe, dass der Spam-Anteil in beiden Teilmengen ~gleich ist.


In [ ]:
# Dein Code hier. (Musterloesung: solution/solution.ipynb)


## 3. Vom Text zu Woertern: Tokenisierung

Naive Bayes arbeitet mit Wort-Wahrscheinlichkeiten — also muessen wir SMS in
Woerter zerlegen. Wir halten es bewusst einfach: Kleinbuchstaben, dann alle
Folgen von Buchstaben/Ziffern extrahieren.

**Aufgabe:** Implementiere `tokenisiere(text)` selbst.

**Tipp:** Nutze das Regex `[a-z0-9']+` und schreibe den Text vorher klein (`.lower()`). **Selbstcheck:** `tokenisiere("WINNER!! Claim your £900 prize now!")` muss `['winner','claim','your','900','prize','now']` ergeben.


In [ ]:
# Dein Code hier. (Musterloesung: solution/solution.ipynb)


## 4. Naive Bayes von Grund auf

Zur Erinnerung (Skript 2.4): Fuer eine SMS mit Woertern $w_1, \dots, w_n$ vergleichen wir

$$P(\text{spam} \mid w_1..w_n) \propto P(\text{spam}) \prod_i P(w_i \mid \text{spam})
\qquad \text{vs.} \qquad
P(\text{ham} \mid w_1..w_n) \propto P(\text{ham}) \prod_i P(w_i \mid \text{ham})$$

Zwei Praxis-Tricks, die du beide selbst umsetzt:

1. **Log-Wahrscheinlichkeiten:** Das Produkt aus hunderten kleinen Zahlen wuerde
   numerisch zu 0 kollabieren (*underflow*). Darum rechnen wir mit Summen von
   Logarithmen: $\log P(c) + \sum_i \log P(w_i \mid c)$ — der Vergleich bleibt derselbe,
   weil der Logarithmus monoton ist.
2. **Laplace-Glaettung:** Ein Wort, das im Training nie in Spam vorkam, haette
   $P(w \mid \text{spam}) = 0$ — ein einziges solches Wort wuerde jede Spam-SMS
   "freisprechen" ($\log 0 = -\infty$). Darum tun wir so, als haetten wir jedes
   bekannte Wort in jeder Klasse **einmal extra** gesehen ($\alpha = 1$):

$$P(w \mid c) = \frac{\text{Anzahl}(w, c) + 1}{\text{Woerter gesamt in } c + |V|}$$

wobei $|V|$ die Groesse des Vokabulars (alle bekannten Woerter) ist.

**Aufgabe:** Baue `trainiere(texte, labels)` (zaehlt pro Klasse die Woerter, das Vokabular und die Priors), `log_wort_wkt(modell, wort, klasse)` (Laplace-Formel oben), `log_posterior(modell, text, klasse)` (Prior + Summe der Log-Wort-Wahrscheinlichkeiten ueber die bekannten Woerter) und `klassifiziere(modell, text)`. **Selbstcheck:** `"URGENT! You have won a free prize, call now!"` → `spam`, `"Ok, see you at the station at 6"` → `ham`.


In [ ]:
# Dein Code hier. (Musterloesung: solution/solution.ipynb)


## 5. Wie gut ist der Filter wirklich?

Jetzt die Bewaehrungsprobe auf den zurueckgehaltenen Testdaten. Neben der
Accuracy schauen wir auf die **Confusion Matrix** und zwei Kennzahlen:

- **Precision** (Spam): Wenn der Filter "Spam" sagt — wie oft stimmt das?
  *(Wichtig: Falsch-Positive = echte SMS im Spamordner = sehr aergerlich!)*
- **Recall** (Spam): Wie viel vom echten Spam faengt der Filter?

**Aufgabe:** Berechne auf den Testdaten `accuracy_score`, `confusion_matrix` und `classification_report`. **Erwartung:** Accuracy ≈ 0,98 — deutlich ueber der "stur ham"-Baseline (≈ 0,87). Schau besonders auf Precision und Recall der Spam-Klasse.


In [ ]:
# Dein Code hier. (Musterloesung: solution/solution.ipynb)


## 6. Was hat das Modell gelernt?

Ein grosser Vorteil von Naive Bayes: Man kann **hineinschauen**. Welche Woerter
sprechen am staerksten fuer Spam? Wir ranken per Log-Verhaeltnis
$\log \frac{P(w \mid \text{spam})}{P(w \mid \text{ham})}$ (nur Woerter, die mindestens 5-mal vorkommen).

**Tipp:** Ranke die Woerter nach dem Log-Verhaeltnis $\log P(w\mid\text{spam}) - \log P(w\mid\text{ham})$ und betrachte nur Woerter, die insgesamt mindestens 5-mal vorkommen.


In [ ]:
# Dein Code hier. (Musterloesung: solution/solution.ipynb)


## 7. Vergleich mit scikit-learn

Zum Abschluss dasselbe Modell mit den Standardwerkzeugen der Praxis:
`CountVectorizer` (zaehlt Woerter) + `MultinomialNB` (genau unser Algorithmus).
Wenn deine Von-Hand-Version gut ist, liegen beide nah beieinander.

**Tipp:** `CountVectorizer(token_pattern=r"[a-z0-9']+", lowercase=True)` + `MultinomialNB(alpha=1.0)`. Deine Von-Hand-Accuracy sollte sehr nah an der scikit-learn-Accuracy liegen.


In [ ]:
# Dein Code hier. (Musterloesung: solution/solution.ipynb)


*(Kleine Abweichungen sind normal: scikit-learn multipliziert mehrfach vorkommende
Woerter pro SMS mit, waehrend Details wie der Umgang mit unbekannten Woertern bei
uns minimal anders geloest sind.)*

## Geschafft — was du jetzt kannst

- einen echten Datensatz laden, explorieren und sauber in Train/Test teilen
- die Bayes-Regel in einen funktionierenden Klassifikator uebersetzen
  (inkl. der zwei Praxis-Tricks: Log-Raum und Laplace-Glaettung)
- ein Modell mit den *richtigen* Metriken bewerten (Precision/Recall statt nur Accuracy)
- dein Von-Hand-Modell gegen eine Industrie-Implementierung benchmarken

**Bonusaufgaben** (optional, ohne Musterloesung):
1. Der Filter steckt manche echte SMS in den Spamordner (Falsch-Positive). Schau dir
   diese SMS an (`test_texte[(vorhersagen_series == "spam") & (test_labels == "ham")]`) — warum stolpert das Modell?
2. Experimentiere mit der Glaettung: Was passiert bei $\alpha = 0{,}01$ oder $\alpha = 10$?
3. Nimm Woerter, die nur 1-mal vorkommen, aus dem Vokabular. Wird das Modell besser oder schlechter — und warum koennte beides passieren?